# Paso 4 — Simulación de los datos
**Artículo:** Wang, Xu & Zhou (2015), *Large Unbalanced Credit Scoring Using Lasso-Logistic Regression Ensemble*.

**Criterio:** los hechos que el artículo reporta se reproducen exactamente y se verifican en una tabla; lo que no reporta se declara como supuesto (README, sección 3.3). La lógica de simulación está en `src/simulate_data.py`; este notebook la ejecuta, la verifica y la visualiza.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import simulate_data as sim

(RAIZ / "data").mkdir(exist_ok=True)
(RAIZ / "results").mkdir(exist_ok=True)
(RAIZ / "figures").mkdir(exist_ok=True)
print("Semilla:", sim.RANDOM_STATE)

Semilla: 42


## 4.1 Generación y almacenamiento del conjunto de datos

In [2]:
df = sim.simular_give_me_some_credit()
df.to_csv(RAIZ / "data" / "datos_simulados.csv", index=False)
print(df.shape)
df.head()

(150000, 11)


,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
0,0.188106,60,1,0.358929,3473.335982,11,2,1,0,1.0,1
1,0.398305,0,0,0.419172,NaN,5,0,3,0,2.0,0
2,0.284392,41,0,0.337692,1280.546332,10,0,1,0,1.0,0
3,0.392191,27,1,0.443356,3974.275025,9,0,1,0,0.0,0
4,0.028050,49,0,5.282542,1688.873862,8,0,2,0,0.0,0


## 4.2 Verificación contra lo reportado
Los valores simulados deben coincidir con los reportados por el artículo. La celda falla (`assert`) si alguno no coincide.

In [3]:
chk = sim.verificar(df)
chk.to_csv(RAIZ / "results" / "paso4_verificacion.csv", index=False)
assert chk["ok"].all(), "La simulación no reproduce lo reportado"
chk

,chequeo,reportado,simulado,ok
0,N total,150000.0,150000.0,True
1,Clase 0,139974.0,139974.0,True
2,Clase 1,10026.0,10026.0,True
3,# predictoras,10.0,10.0,True
4,age min,0.0,0.0,True
5,age max,109.0,109.0,True
6,age media,52.3,52.3,True


## 4.3 Supuestos: valores faltantes y códigos especiales
`MonthlyIncome` ≈ 20 % y `NumberOfDependents` ≈ 2,6 % de valores faltantes; ≈ 0,18 % de filas con código 96/98 en las tres variables de atraso. Estos códigos se usan en el Paso 5, donde se generan banderas para valores especiales, como describe el artículo.

In [4]:
faltantes = (df.isna().mean() * 100).round(2).rename("% faltantes").to_frame()
faltantes.to_csv(RAIZ / "results" / "paso4_faltantes.csv")
atrasos = ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTimes90DaysLate", "NumberOfTime60-89DaysPastDueNotWorse"]
n_esp = int((df[atrasos] >= 90).any(axis=1).sum())
print(f"Filas con código especial (96/98): {n_esp} ({n_esp/len(df)*100:.2f}%)")
faltantes

Filas con código especial (96/98): 262 (0.17%)


,% faltantes
RevolvingUtilizationOfUnsecuredLines,0.00
age,0.00
NumberOfTime30-59DaysPastDueNotWorse,0.00
DebtRatio,0.00
MonthlyIncome,19.83
NumberOfOpenCreditLinesAndLoans,0.00
NumberOfTimes90DaysLate,0.00
NumberRealEstateLoansOrLines,0.00
NumberOfTime60-89DaysPastDueNotWorse,0.00
NumberOfDependents,2.62


## 4.4 Relación con el impago (supuesto de magnitud)
El artículo indica únicamente la dirección de la relación: más atrasos y mayor utilización del crédito implican mayor impago, mayor ingreso implica menor impago y el número de dependientes aporta poca información. La magnitud es un supuesto de la simulación, documentado en la tabla de medias por clase. Las medias de las variables de atraso incluyen los códigos 96/98.

In [5]:
medias = df.groupby(sim.TARGET).mean(numeric_only=True).T.round(3)
medias.columns = ["clase 0 (no impago)", "clase 1 (impago)"]
medias["razón 1/0"] = (medias.iloc[:, 1] / medias.iloc[:, 0]).round(2)
medias.to_csv(RAIZ / "results" / "paso4_medias_por_clase.csv")
medias

,clase 0 (no impago),clase 1 (impago),razón 1/0
RevolvingUtilizationOfUnsecuredLines,0.226,0.322,1.42
age,52.425,50.557,0.96
NumberOfTime30-59DaysPastDueNotWorse,0.318,1.227,3.86
DebtRatio,0.904,1.278,1.41
MonthlyIncome,5330.459,4083.779,0.77
NumberOfOpenCreditLinesAndLoans,8.488,7.351,0.87
NumberOfTimes90DaysLate,0.220,0.797,3.62
NumberRealEstateLoansOrLines,0.999,0.780,0.78
NumberOfTime60-89DaysPastDueNotWorse,0.210,0.608,2.90
NumberOfDependents,0.765,0.752,0.98


In [6]:
vars_graf = ["RevolvingUtilizationOfUnsecuredLines", "age", "DebtRatio", "MonthlyIncome",
             "NumberOfTime30-59DaysPastDueNotWorse", "NumberOfOpenCreditLinesAndLoans"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, v in zip(axes.ravel(), vars_graf):
    x = df[[v, sim.TARGET]].dropna()
    lo, hi = x[v].quantile([0.0, 0.99])
    bins = np.linspace(lo, hi, 40)
    for c, col in [(0, "#3b6ea5"), (1, "#c0392b")]:
        ax.hist(x.loc[x[sim.TARGET] == c, v].clip(lo, hi), bins=bins, density=True,
                alpha=0.55, color=col, label=f"clase {c}")
    ax.set_title(v, fontsize=9)
axes[0, 0].legend()
fig.suptitle("Distribución por clase de variables simuladas (recortadas al percentil 99)")
fig.tight_layout()
fig.savefig(RAIZ / "figures" / "paso4_distribuciones_por_clase.png", dpi=130)
print("figura guardada")

figura guardada


## 4.5 Resumen y limitaciones
**Reproducido exactamente:** tamaño de la muestra, conteos de cada clase, número de variables y mínimo, máximo y media de `age`.

**Supuestos:** forma de la distribución de cada variable, desviación estándar de `age`, porcentaje de valores faltantes, códigos 96/98, magnitud del efecto de cada variable sobre el impago e independencia entre predictoras dada la clase.

**Limitación principal:** la intensidad de la señal es un supuesto. Por eso no se esperan AUC absolutos iguales a los del artículo y el Paso 6 compara patrones cualitativos.

**Reproducibilidad:** `RANDOM_STATE = 42`; dos ejecuciones generan el mismo CSV.